This Notebook can be run on a corpus to map all non-BMP unicode characters to characters within the unicode BMP.

In [25]:
import re
import os
NON_BMP = re.compile(r"[^\u0000-\uFFFF]")
PRIVATE = re.compile(r"[\uE000-\uF8FF]")
charmap = {}
priv_iter = 0xE000 #Start at the first private unicode character

In [ ]:
folder = r"CORPUS_DIR"

I'm unsure if this is strictly necessary, but it might be better to have this than not have this;
Most of the language that the mismatcher trip up are Chinese languages, likely because they use rare or archaic characters not in the BMP (especially so because LZH is classical Chinese and it runs into a lot of errors)
To maintain consistency I'm maintaining a dictionary to consistently map the same characters to the same private unicode characters.
Code downstream might give different results if I were to go with a naive approach that maps every character onto the same blank character, or that maps every newly encountered character onto a new blank character.

In [ ]:
def find_and_replace(text: str, path):
    global priv_iter

    if not re.search(NON_BMP, text):
        return text

    for match in re.finditer(NON_BMP, text):
        ch = match.group()
        print(f"Found incompatible character {ch} in {path}.")
        if ch not in charmap:
            if priv_iter > 0xF8FF:
                raise ValueError("Reached end of private unicode characters! Aborting...")
            print(f"Found incompatible character {ch} in {path}.")
            charmap[ch] = chr(priv_iter)
            priv_iter += 1
        text = text.replace(ch, charmap[ch])

    return text

In [28]:
for filename in os.listdir(folder):
        if filename.endswith('.txt'):
            path = os.path.join(folder, filename)
            with open(path, 'r', encoding='utf-8') as f:
                text = f.read()
            with open(path, 'w', encoding='utf-8') as f:
                f.write(find_and_replace(text, path))